# The Ninefold Tide — E101 CAM03B private Kaggle Blender notebook (GitHub upload)

The original v11 source code is preserved. This private notebook uses NO GPU hours or large downloads until the real owner-only v11 source is present in Kaggle Inputs.

The public GitHub repository stores code only, never character drawings or original ZIPs.

[Authoritative S1 Drive sources](https://drive.google.com/drive/folders/1jAWl-mMcE-EDeQR8vHv1B1RqNnGDpuC1).


In [ ]:
from pathlib import Path
import json

INPUT_DIR = Path('/kaggle/input')
BUNDLE_NAMES = (
    'Ninefold_S1_Private_Kaggle_Kit_v11.zip',
    'Ninefold_S1_Private_Kaggle_Kit_v10.zip',
    'Ninefold_S1_Private_Kaggle_Kit_v09r1.zip',
)
ORIGINAL_V07 = 'NINEFOLD_2D_GREASE_PENCIL_KAGGLE_TRANSFER_v07.zip'
ORIGINAL_V08 = 'The_Ninefold_Tide_E101_CAM03B_v08_Source.zip'

attached = []
if INPUT_DIR.exists():
    attached = [p for p in INPUT_DIR.rglob('*') if p.is_file()]
NINEFOLD_SOURCE_READY = (any(p.name in BUNDLE_NAMES for p in attached) or
    (any(p.name == ORIGINAL_V07 for p in attached) and any(p.name == ORIGINAL_V08 for p in attached)))
UPLOAD_STAGE = 'SOURCE_READY' if NINEFOLD_SOURCE_READY else 'STAGED_NEEDS_PRIVATE_SOURCE'
print('NINEFOLD_E101_UPLOAD_STAGE:', UPLOAD_STAGE)
if not NINEFOLD_SOURCE_READY:
    print('Your private Kaggle notebook was uploaded. To enable the native Blender build, attach the owner-only')
    print('Ninefold_S1_Private_Kaggle_Kit_v11.zip as a PRIVATE Kaggle dataset in Add Input, then rerun.')
    print('Source: https://drive.google.com/file/d/1_Q69H3rJleFiuKKnxgq-HoFdI8hGeGK9/view')
    print('No Blender download, no GPU inference, no video creation, and no source data is published.')
    stage_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    (stage_dir/'ninefold_e101_upload_status.json').write_text(
        json.dumps({'status': UPLOAD_STAGE, 'source_present': False,
        'private_notebook_uploaded': True, 'artist_approved': False,
        'video_created': False, 'gpu_required_to_stage': False}, indent=2))


# The Ninefold Tide — Season 1 E101 CAM03B Blender reality check (v11)

**Free, private, no paid credits.** Builds a *real editable Blender stage* from the canonical **v05 world → v07 2D Grease Pencil actors → v08 Naya gallery camera and rail**. Does not create original 2D character drawings, lip-sync, an animatic, or a finished video. **Do not approve the art based on a technically successful build.**

## Start

1. Import this notebook into **Kaggle Notebooks**, choose a **private** notebook.
2. Under **Add Input**, attach `Ninefold_S1_Private_Kaggle_Kit_v11.zip` as your own private dataset (around 2 MB), or provide separately the v07 and v08 source ZIPs from the project's Google Drive ART folders.
3. Run in order. It does not touch ASMR or other Kaggle projects.
4. The only possible major remote download is **Blender 4.5.15 LTS (378 MB)**, and only if Blender 4.5+ is not already available. It downloads inside Kaggle, **not onto your phone**. If Kaggle Internet is disabled, it stops instead of silently installing Blender 4.0.2. You may attach a private reusable Blender 4.5 distribution dataset to avoid repeat downloads.

Assets: [v07 original ZIP](https://drive.google.com/file/d/1dSFNAqlswiGU2tAWrZi85T1y9mvdu5x3/view), [v08 complete source ZIP](https://drive.google.com/file/d/1ujsv99EHB7VhXDtAzx5XfzCr9DkG2kIg/view), [director's cut](https://docs.google.com/document/d/1owa_iY7Y9jeF7NHcVvKUbhJPpV-wyQ8YVR00ZbRabrs/edit).

In [ ]:
if NINEFOLD_SOURCE_READY:
    from pathlib import Path, PurePosixPath
    import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile, re, urllib.request
    BASE=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    ROOT=BASE/'ninefold_s1_v11'
    ROOT.mkdir(parents=True, exist_ok=True)
    INPUTS=[p for p in (Path('/kaggle/input'), BASE) if p.exists()]
    V07_NAME='NINEFOLD_2D_GREASE_PENCIL_KAGGLE_TRANSFER_v07.zip'
    V08_NAME='The_Ninefold_Tide_E101_CAM03B_v08_Source.zip'
    KIT_NAMES=('Ninefold_S1_Private_Kaggle_Kit_v11.zip', 'Ninefold_S1_Private_Kaggle_Kit_v10.zip', 'Ninefold_S1_Private_Kaggle_Kit_v09r1.zip', 'Ninefold_S1_Private_Kaggle_Kit_v09.zip')
    V07_EXPECTED='98c5f655860f53d68019605955f3c5e3951b139eb1570153f9fb1cf1a7655e91'
    V05_EXPECTED='24c0fb2ae611b4ba50792eb20abc6e192df016401dfa94e0b1eef452dfb77d2c'
    def hash_file(p):
     h=hashlib.sha256()
     with open(p,'rb') as f:
      for block in iter(lambda:f.read(1<<20), b''):h.update(block)
     return h.hexdigest()
    def files_named(name):
     result=[]
     for root in INPUTS:
      result.extend(q for q in root.rglob(name) if q.is_file())
     return result

    def unpack_zip(p, dest):
     dest.mkdir(parents=True,exist_ok=True)
     with zipfile.ZipFile(p) as z:
      if z.testzip():raise ValueError('Corrupt ZIP: '+str(p))
      for m in z.infolist():
       n=PurePosixPath(m.filename.replace(chr(92),'/'))
       if n.is_absolute() or '..' in n.parts:raise ValueError('Unsafe ZIP member')
       if m.file_size > 256_000_000:raise ValueError('Unexpected oversized ZIP member')
      z.extractall(dest)
     return dest
    kit=next((found for name in KIT_NAMES if (found:=files_named(name))), [])
    if kit:
     print('Located private consolidated kit:',kit[0]);unpack_zip(kit[0],ROOT/'kit')
     sources=[ROOT/'kit',*INPUTS]
    else:sources=INPUTS

    def locate(name):
     for source in sources:
      for x in source.rglob(name):
       if x.is_file():return x
     raise FileNotFoundError('Missing source '+name+'. Attach 2 MB private kit to Kaggle Input.')
    v07=locate(V07_NAME);v08=locate(V08_NAME)
    assert hash_file(v07)==V07_EXPECTED, 'v07 SHA differs from verified original source'
    S7=unpack_zip(v07,ROOT/'v07')
    foundation=S7/'NINEFOLD_Production_Foundation_v05_Blender_GLTF_Assets.zip'
    assert hash_file(foundation)==V05_EXPECTED, 'v05 nested foundation archive altered'
    S8=unpack_zip(v08,ROOT/'v08')
    unpack_zip(foundation,ROOT/'foundation')
    WORLD=ROOT/'foundation'/'Ninefold_Production_Foundation_v05'
    assert (WORLD/'blender'/'BUILD_VEYRA_SCENE_BLENDER_45.py').is_file()
    assert (S7/'BUILD_2D_GREASE_PENCIL_WORKSPACE_v07.py').is_file()
    assert (S8/'blender'/'BUILD_EP101_CAM03B_RAIL_CONTACT_v08.py').is_file()
    print('Source integrity: PASS. Files in:',ROOT)


## Blender 4.5 LTS without a paid subscription

The **existing older Kaggle apt Blender 4.0.2 is not enough** for the actual Grease Pencil v3 builder. If necessary, the following cell obtains the official portable Linux release in Kaggle itself, verifies the Blender.org checksum and reuses an attached archive if one is available. The optional download is approximately 378 MB and occurs on Kaggle's server. On a metered connection, attach a previously cached Blender 4.5 package instead of downloading it again.

In [ ]:
if NINEFOLD_SOURCE_READY:
    ALLOW_REMOTE_BLENDER_DOWNLOAD=True # Kaggle server only, no paid credits, ~378 MB if not cached
    BLENDER_ARCHIVE='blender-4.5.15-linux-x64.tar.xz'
    BLENDER_URL='https://download.blender.org/release/Blender4.5/'+BLENDER_ARCHIVE
    BLENDER_SHA_URL='https://download.blender.org/release/Blender4.5/blender-4.5.15.sha256'

    def version(p):
     try:
      q=subprocess.run([str(p),'--version'],capture_output=True,text=True,timeout=30)
      m=re.search(r'Blender (\d+)\.(\d+)',q.stdout)
      return tuple(map(int,m.groups())) if q.returncode==0 and m else (0,0)
     except (OSError,subprocess.TimeoutExpired):return (0,0)

    blender=shutil.which('blender')
    if blender and version(blender)<(4,5):
     print('System Blender',version(blender),'is too old; searching portable Blender 4.5')
     blender=None
    if not blender:
     for root in INPUTS+[ROOT]:
      options=list(root.rglob('blender-4.5*/blender')) if root.exists() else []
      blender=next((str(p) for p in options if p.is_file() and version(p)>=(4,5)),None)
      if blender:break
    if not blender:
     archive=next(iter(files_named(BLENDER_ARCHIVE)),None)
     if archive is None:
      if not ALLOW_REMOTE_BLENDER_DOWNLOAD:
       raise RuntimeError('Blender 4.5+ unavailable. Set ALLOW_REMOTE_BLENDER_DOWNLOAD=True or attach cached portable Linux Blender 4.5 archive.')
      archive=ROOT/BLENDER_ARCHIVE
      if not archive.exists():
       print('Downloading official Blender 4.5.15 (~378 MB) on remote Kaggle, no paid API')
       try:
        with urllib.request.urlopen(BLENDER_URL,timeout=90) as src,open(archive,'wb') as dst:
         shutil.copyfileobj(src,dst,1024*1024)
       except Exception as exc:
        raise RuntimeError('Kaggle Internet unavailable; attach Blender 4.5 portable release as a private Input') from exc
     # Obtain the official SHA from Blender.org; never trust a wrong or truncated download.
     try:
      with urllib.request.urlopen(BLENDER_SHA_URL,timeout=60) as src:manifest=src.read().decode('utf-8')
      match=next((line for line in manifest.splitlines() if BLENDER_ARCHIVE in line),None)
      if not match:raise ValueError('Official sha file has no entry for archive')
      expected=re.match(r'^([0-9a-fA-F]{64})',match).group(1).lower()
      if hash_file(archive)!=expected:raise ValueError('Blender archive checksum mismatch')
     except Exception as exc:
      raise RuntimeError('Cannot verify official Blender SHA256. Refusing to execute package.') from exc
     install=ROOT/'portable_blender';install.mkdir(exist_ok=True)
     with tarfile.open(archive,'r:xz') as tar:
      # Contained only in /kaggle/working; reject archive path traversals or links.
      for m in tar.getmembers():
       n=PurePosixPath(m.name)
       if n.is_absolute() or '..' in n.parts:
        raise ValueError('Unsafe portable Blender tar member')
      tar.extractall(install,filter='data')
     blender=next((str(p) for p in install.rglob('blender') if p.is_file() and version(p)>=(4,5)),None)
    if not blender or version(blender)<(4,5):raise RuntimeError('Blender 4.5 executable still not available')
    print('Using verified Blender:',blender,'version',version(blender))


## Build exact original source sequence

Only updates a new private workspace. The original v05/v07 archives remain unchanged; no automatic bulk episode rendering is performed.

In [ ]:
if NINEFOLD_SOURCE_READY:
    def run_blender(args,timeout=850):
     cmd=[str(blender),'--background',*map(str,args)]
     res=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout)
     print(res.stdout[-11000:])
     if res.returncode:raise RuntimeError('Blender stage exited '+str(res.returncode)+'; review logged error')

    original=WORLD/'blender'/'BUILD_VEYRA_SCENE_BLENDER_45.py'
    script=original.read_text(encoding='utf-8')
    script=script.replace("scene.view_settings.view_transform='AgX'", "try: scene.view_settings.view_transform='AgX'\nexcept Exception: scene.view_settings.view_transform='Standard'")
    patched=WORLD/'blender'/'BUILD_VEYRA_SCENE_KAGGLE_v11.py'
    patched.write_text(script,encoding='utf-8');compile(script,str(patched),'exec')
    run_blender(['--factory-startup','--python',patched],timeout=850)
    base=WORLD/'NINEFOLD_EP101_2p5D_WORLD_v05.blend'
    assert base.is_file(), 'v05 builder did not create editable .blend'
    O7=ROOT/'v07_output';O7.mkdir(exist_ok=True)
    # Blender 4.5 can create a default GP layer from type=EMPTY. Do NOT edit the source archive;
    # remove the implicit layer in a copy so the desired output is exactly nine layers.
    original7=(S7/'BUILD_2D_GREASE_PENCIL_WORKSPACE_v07.py').read_text(encoding='utf-8')
    marker7='    for layer_name in LAYER_NAMES:'
    assert original7.count(marker7)==1, 'v07 builder changed: do not apply unverified patch'
    patched7=original7.replace(marker7,
        '    for _default_gp_layer in tuple(drawing.data.layers):\n'
        '        drawing.data.layers.remove(_default_gp_layer)\n'
        +marker7)
    patched7_path=ROOT/'BUILD_GP3_NINE_LAYERS_v11.py'
    patched7_path.write_text(patched7,encoding='utf-8');compile(patched7,str(patched7_path),'exec')
    run_blender([base,'--python',patched7_path,'--',O7],timeout=650)
    blend7=O7/'NINEFOLD_EP101_2D_GP_CHARACTERS_3D_BACKGROUND_v07.blend'
    qa7=O7/'NINEFOLD_v07_Blender_Runtime_QA.json'
    assert blend7.is_file() and qa7.is_file(), 'v07 GP scene not created'
    data7=json.loads(qa7.read_text())
    assert len(data7['characters'])==3 and all(x['layers']==9 for x in data7['characters'])
    assert not data7['approved_final_animation']
    O8=ROOT/'v08_output';O8.mkdir(exist_ok=True)
    # Keep Naya's blank 2D drawing plane facing the active CAM03B perspective, not CAM01.
    original8=(S8/'blender'/'BUILD_EP101_CAM03B_RAIL_CONTACT_v08.py').read_text(encoding='utf-8')
    marker8='scene.camera=cam\n'
    assert original8.count(marker8)==1, 'v08 builder changed: do not apply unverified camera patch'
    patch8='''# Keep blank 2D drawing plane facing the locked camera (no character art).
    _heading=Vector(cam.location)-Vector(root.location)
    drawing.rotation_euler=(math.pi/2,0,math.atan2(_heading.y,_heading.x)-math.pi/2)
    '''
    patched8=original8.replace(marker8,marker8+patch8)
    patched8_path=ROOT/'BUILD_CAM03B_GP_PLANE_v11.py'
    patched8_path.write_text(patched8,encoding='utf-8');compile(patched8,str(patched8_path),'exec')
    run_blender([blend7,'--python',patched8_path,'--',O8],timeout=650)
    blend8=O8/'NINEFOLD_E101_CAM03B_2D_CHARACTER_3D_WORLD_v08.blend'
    qa8=O8/'NINEFOLD_CAM03B_BLENDER_RUNTIME_REPORT.json'
    assert blend8.is_file() and qa8.is_file(), 'v08 physical camera stage not created'
    data8=json.loads(qa8.read_text())
    assert data8['new_rail_meshes']>=17 and data8['grease_pencil_layers']==9
    assert data8['camera']=='CAM_03B_NAYA_RAIL_CONTACT'
    assert all(abs(a-b)<0.01 for a,b in zip(data8['foot'], (2.1,-3.17,3.355)))
    assert all(abs(a-b)<0.01 for a,b in zip(data8['grip'], (2.36,-2.685,4.16)))
    assert abs(data8['height_m']-1.64)<0.001 and not data8['actual_2d_key_drawings_approved']
    print('ACTUAL BLENDER TECHNICAL BUILD: PASS; CREATIVE ART/ANIMATION: UNAPPROVED')
    print('EDITABLE BLEND:',blend8,'size bytes',blend8.stat().st_size)
    print('RUNTIME REPORT:',qa8)


In [ ]:
if NINEFOLD_SOURCE_READY:
    # A deliberately limited internal 3D camera/occlusion still: not character artwork.
    RENDER_TECHNICAL_LAYOUT_STILL=True
    tech_preview=ROOT/'v11_output'/'E101_CAM03B_Blender_Camera_Occlusion_UNAPPROVED.png'
    if RENDER_TECHNICAL_LAYOUT_STILL:
        tech_preview.parent.mkdir(exist_ok=True, parents=True)
        render_script=ROOT/'render_cam03b_technical_only.py'
        render_script.write_text("""import bpy
    scene=bpy.context.scene
    scene.render.engine='CYCLES'
    scene.cycles.samples=8
    scene.render.resolution_x=1280
    scene.render.resolution_y=720
    scene.render.resolution_percentage=50
    scene.render.image_settings.file_format='PNG'
    scene.frame_set(1)
    scene.render.filepath=r'%s'
    bpy.ops.render.render(write_still=True)
    """ % str(tech_preview), encoding='utf-8')
        run_blender([blend8, '--python', render_script],timeout=850)
        if not tech_preview.is_file() or tech_preview.stat().st_size<5000:
            raise RuntimeError('No nonempty internal Blender technical camera still saved')
    print('Camera technical layout still:',str(tech_preview) if tech_preview.exists() else 'disabled')
    print('This still contains NO approved drawn Naya; no video is authored.')


In [ ]:
if NINEFOLD_SOURCE_READY:
    # Write a truthful manifest; no video is generated, no director approval is invented.
    manifest={
     'project':'The Ninefold Tide','episode':101,'shot':'CAM03B',
     'technical_blender_runtime':'PASS',
     'source_v07_sha256':hash_file(v07),'source_v08_sha256':hash_file(v08),
     'blender_version':version(blender),
     'editable_blend_file':str(blend8),
     'v11_internal_3d_camera_still':str(tech_preview) if tech_preview.is_file() else None,
     'art_approval':False,'actor_keyframes_approved':False,'audio_approved':False,
     'release_approved':False,
     'scene_101_original_drawing_present':False,
     'remaining':['actual hand-drawn Naya facial/hand poses','finished painted BG','real grip and contact artwork','human art review','timed acted animation & natural mix']
    }
    report=ROOT/'NINEFOLD_E101_CAM03B_v11_RUNTIME_RESULT.json'
    report.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('RESULT:',report)
    print('Download/edit output from Kaggle working; back up sources with your Google Drive ART project. Do NOT publish a technical greybox as finished animation.')
